Project : Analyzing the Trends of COVID-19 with Python

                                                      Submitted by,
                                                      Aksa Anna Jose
                                                      

In [1]:
pip install prophet plotly pandas

In [2]:
import pandas as pd
import plotly.graph_objects as go
from prophet import Prophet
import numpy as np
import matplotlib as plt
import seaborn as sns
#import warnings
#warnings.filterwarnings('ignore')


In [3]:
# Load the Dataset
df = pd.read_csv('/content/Covid_19_Clean_Complete.csv')

In [4]:
# Convert Date column to datetime format
df['Date'] = pd.to_datetime(df['Date'])

In [11]:
# Group by Date to get daily global/national aggregates
daily_data = df.groupby('Date').sum().reset_index()

In [6]:
# Display the first few rows to verify
print(daily_data.head())

        Date                                     Province/State  \
0 2020-01-22  Australian Capital TerritoryNew South WalesNor...   
1 2020-01-23  Australian Capital TerritoryNew South WalesNor...   
2 2020-01-24  Australian Capital TerritoryNew South WalesNor...   
3 2020-01-25  Australian Capital TerritoryNew South WalesNor...   
4 2020-01-26  Australian Capital TerritoryNew South WalesNor...   

                                      Country/Region         Lat         Long  \
0  AfghanistanAlbaniaAlgeriaAndorraAngolaAntigua ...  5594.20365  6140.869714   
1  AfghanistanAlbaniaAlgeriaAndorraAngolaAntigua ...  5594.20365  6140.869714   
2  AfghanistanAlbaniaAlgeriaAndorraAngolaAntigua ...  5594.20365  6140.869714   
3  AfghanistanAlbaniaAlgeriaAndorraAngolaAntigua ...  5594.20365  6140.869714   
4  AfghanistanAlbaniaAlgeriaAndorraAngolaAntigua ...  5594.20365  6140.869714   

   Confirmed  Deaths  Recovered  Active  \
0        555      17         28     510   
1        654      18    

In [7]:
# Calculate Active Cases if not already present
daily_data['Active'] = daily_data['Confirmed'] - daily_data['Recovered'] - daily_data.get('Deaths', 0)

# Create an interactive line plot
fig = go.Figure()

fig.add_trace(go.Scatter(x=daily_data['Date'], y=daily_data['Confirmed'], name='Confirmed Cases', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=daily_data['Date'], y=daily_data['Recovered'], name='Recovered Cases', line=dict(color='green')))
fig.add_trace(go.Scatter(x=daily_data['Date'], y=daily_data['Active'], name='Active Cases', line=dict(color='orange')))

fig.update_layout(
    title='COVID-19 Global/India Trend Analysis',
    xaxis_title='Date',
    yaxis_title='Number of Cases',
    template='plotly_white'
)

fig.show()

In [8]:
# Prepare data specifically for Prophet
prophet_df = daily_data[['Date', 'Confirmed']].rename(columns={'Date': 'ds', 'Confirmed': 'y'})

# Initialize the Prophet model
model = Prophet(yearly_seasonality=False, weekly_seasonality=True, daily_seasonality=False)
model.fit(prophet_df)

# Create a dataframe for future predictions (7 days into the future)
future = model.make_future_dataframe(periods=7)

# Generate predictions
forecast = model.predict(future)

# Display the forecasted values for the next week
print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(7))

            ds          yhat    yhat_lower    yhat_upper
188 2020-07-28  1.632021e+07  1.621109e+07  1.642660e+07
189 2020-07-29  1.652998e+07  1.642870e+07  1.663470e+07
190 2020-07-30  1.674392e+07  1.663623e+07  1.685070e+07
191 2020-07-31  1.695911e+07  1.684918e+07  1.706564e+07
192 2020-08-01  1.716677e+07  1.706204e+07  1.728011e+07
193 2020-08-02  1.736430e+07  1.724895e+07  1.747550e+07
194 2020-08-03  1.755889e+07  1.744259e+07  1.767829e+07


In [9]:
# Create an interactive plot combining actual data and predictions
fig_forecast = go.Figure()

# Plot historical actual data
fig_forecast.add_trace(go.Scatter(x=prophet_df['ds'], y=prophet_df['y'], mode='markers+lines', name='Actual Confirmed', line=dict(color='blue')))

# Plot forecasted data (including the future 7 days)
fig_forecast.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat'], mode='lines', name='Predicted Trend', line=dict(color='red', dash='dash')))

# Plot upper and lower uncertainty bounds
fig_forecast.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat_upper'], mode='lines', name='Upper Bound', line=dict(width=0), showlegend=False))
fig_forecast.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat_lower'], mode='lines', name='Lower Bound', line=dict(width=0), fill='tonexty', fillcolor='rgba(255, 0, 0, 0.1)', showlegend=False))

fig_forecast.update_layout(
    title='7-Day COVID-19 Confirmed Cases Forecast',
    xaxis_title='Date',
    yaxis_title='Number of Cases',
    template='plotly_white'
)

fig_forecast.show()